In [ ]:
import sys
print(sys.executable)

In [1]:
from presidio_analyzer import AnalyzerEngine

# Intialize the analyzer
analyzer = AnalyzerEngine()

# Sample text with multiple PII types
text = """
Hi, my name is John Smith and I live in Seattle.
My email is johm.smith@example.com and my phone
number is 206-555-0147. My SSN is 123-456-6789
and my credit card is 4111-1111-1111-1111
"""

# Analyze the text
results = analyzer.analyze(text=text, language="en")

# Print what was found
for result in results:
    print(f"{result.entity_type}: '{text[result.start:result.end].strip()}' "
    f"(score: {result.score:.2f}, position: {result.start}-{result.end})")

EMAIL_ADDRESS: 'johm.smith@example.com' (score: 1.00, position: 62-84)
UK_NHS: '206-555-0147' (score: 1.00, position: 108-120)
CREDIT_CARD: '4111-1111-1111-1111' (score: 1.00, position: 167-186)
PERSON: 'John Smith' (score: 0.85, position: 16-26)
LOCATION: 'Seattle' (score: 0.85, position: 41-48)
PHONE_NUMBER: '206-555-0147' (score: 0.75, position: 108-120)
PHONE_NUMBER: '123-456-6789' (score: 0.75, position: 132-144)
URL: 'johm.sm' (score: 0.50, position: 62-69)
URL: 'example.com' (score: 0.50, position: 73-84)


In [2]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

text = "My name is John Smith and my email is john.smith@example,com"

# Detect PII
results = analyzer.analyze(text=text, language="en")

# Anonymize with default settings (replaces with entity type lables)
anonymized = anonymizer.anonymize(text=text, analyzer_results=results)

print(anonymized.text)

My name is <PERSON> and my email is <URL>ith@example,com


# Limitation: Presidio is English-first
Presidio's default setup uses an English spaCy model ('en_core_web_lg') and recognizers
tuned for English/US formats. Below, the same kind of sentence (name, city, phone number)
is analyzed in English, Spanish, French and Yoruba

In [3]:
samples = {
    "English: My name is John Smith, I live in London and my phone number is 212-555-0918"
    "Spanish": "Me llamo Juan García, vivo en Madrid y mi número de teléfono es 612 345 678.",
    "French":  "Je m'appelle Marie Dubois, j'habite à Paris et mon numéro est 06 12 34 56 78.",
    "Yoruba":  "Orúkọ mi ni Adébáyọ̀ Olúwaṣeun, mo ń gbé ní Ìbàdàn, nọ́mbà fóònù mi ni 0803 123 4567.",
}

for lang, text in samples.items():
    print(f"\n{lang}\n{text}")
    results = analyzer.analyze(text=text, language="en")
    if not results:
        print("(nothing detected)")
    for result in sorted(results, key=lambda r: r.start):
        print(f"{result.entity_type}: '{text[result.start:result.end].strip()}'"
              f"(score: {result.score:2f}, position: {result.start}-{result.end})")


English: My name is John Smith, I live in London and my phone number is 212-555-0918Spanish
Me llamo Juan García, vivo en Madrid y mi número de teléfono es 612 345 678.
PERSON: 'Juan García'(score: 0.850000, position: 9-20)

French
Je m'appelle Marie Dubois, j'habite à Paris et mon numéro est 06 12 34 56 78.
PERSON: 'Je m'appelle'(score: 0.850000, position: 0-12)
PERSON: 'Marie Dubois'(score: 0.850000, position: 13-25)
PHONE_NUMBER: '06 12 34 56 78'(score: 0.400000, position: 62-76)
DATE_TIME: '56 78'(score: 0.850000, position: 71-76)

Yoruba
Orúkọ mi ni Adébáyọ̀ Olúwaṣeun, mo ń gbé ní Ìbàdàn, nọ́mbà fóònù mi ni 0803 123 4567.
PHONE_NUMBER: '0803 123 4567'(score: 0.400000, position: 71-84)
